# SafeLink — Manual URL Tester
Run cells **top to bottom once** to load all models, then use the last two cells to test URLs.

> **CNN:** Uses the real `safelink_model.tflite` via `ai_edge_litert` — results now match the Android app.  
> Install once: `pip install ai-edge-litert`  
> URLBert (ONNX) and the Fusion Engine are identical to the Android app.

In [3]:
# ── Cell 1: Imports ────────────────────────────────────────────────
import os, sys, math, json, time
import re as _re
import numpy as np
import onnxruntime as ort
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib'], check=True)

# Real TFLite runtime -- try ai_edge_litert (Python 3.13+), then tflite_runtime (3.8-3.12)
try:
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    _TFLITE_BACKEND = 'ai_edge_litert'
except ImportError:
    try:
        import tflite_runtime.interpreter as _tflite  # type: ignore[reportMissingImports]
        TFLiteInterpreter = _tflite.Interpreter
        _TFLITE_BACKEND = 'tflite_runtime'
    except ImportError:
        TFLiteInterpreter = None
        _TFLITE_BACKEND = None

ML_DIR = os.path.abspath('.')
if ML_DIR not in sys.path:
    sys.path.insert(0, ML_DIR)

from feature_extractor import extract_feature_vector, FEATURE_COLUMNS

print('Python     :', sys.version.split()[0])
print('ONNX RT    :', ort.__version__)
print('NumPy      :', np.__version__)
print('Features   :', len(FEATURE_COLUMNS))
print('TFLite     :', _TFLITE_BACKEND or 'NOT AVAILABLE -- install ai-edge-litert')

Python     : 3.12.0
ONNX RT    : 1.25.0
NumPy      : 2.4.4
Features   : 36
TFLite     : ai_edge_litert


In [4]:
# ── Cell 2: Load scaler ────────────────────────────────────────────
with open('models/scaler_params.json') as f:
    sp = json.load(f)

mean_  = np.array(sp['mean'],  dtype=np.float32)
scale_ = np.array(sp['scale'], dtype=np.float32)

def scale_features(vec):
    return (np.array(vec, dtype=np.float32) - mean_) / scale_

print(f'Scaler loaded  ({len(mean_)} features)')

Scaler loaded  (36 features)


In [5]:
# ── Cell 3: Load URLBert ONNX + WordPiece tokenizer ───────────────
MAX_LEN    = 64
VOCAB_FILE = '../urlbert-tiny-v4-phishing-classifier/vocab.txt'
ONNX_FILE  = 'models/urlbert.onnx'

vocab = {}
with open(VOCAB_FILE, encoding='utf-8') as f:
    for idx, line in enumerate(f):
        vocab[line.strip()] = idx
print(f'Vocab loaded   ({len(vocab)} tokens)')

sess = ort.InferenceSession(ONNX_FILE)
IN0  = sess.get_inputs()[0].name
IN1  = sess.get_inputs()[1].name
print(f'URLBert loaded ({ONNX_FILE})')

# ── Tokenizer (BertTokenizer, do_lower_case=True) ─────────────────
def _is_punct(ch):
    c = ord(ch)
    return c in range(33,48) or c in range(58,65) or c in range(91,97) or c in range(123,127)

def _basic_tokenize(text):
    tokens, buf = [], []
    for ch in text:
        if ch.isspace():
            if buf: tokens.append(''.join(buf)); buf = []
        elif _is_punct(ch):
            if buf: tokens.append(''.join(buf)); buf = []
            tokens.append(ch)
        else:
            buf.append(ch)
    if buf: tokens.append(''.join(buf))
    return tokens

def _wordpiece(word):
    unk = vocab.get('[UNK]', 1)
    if word in vocab: return [vocab[word]]
    ids, start = [], 0
    while start < len(word):
        end, found = len(word), False
        prefix = '' if start == 0 else '##'
        while end > start:
            sub = prefix + word[start:end]
            if sub in vocab:
                ids.append(vocab[sub]); start = end; found = True; break
            end -= 1
        if not found: return [unk]
    return ids

def _tokenize(url):
    ids  = [0] * MAX_LEN
    mask = [0] * MAX_LEN
    ids[0]  = vocab.get('[CLS]', 2); mask[0] = 1
    pos = 1
    for word in _basic_tokenize(url.lower()):
        for wid in _wordpiece(word):
            if pos >= MAX_LEN - 1: break
            ids[pos] = wid; mask[pos] = 1; pos += 1
        if pos >= MAX_LEN - 1: break
    ids[pos]  = vocab.get('[SEP]', 3); mask[pos] = 1
    return ids, mask

def bert_score(url):
    url = url.rstrip('/') or url  # strip trailing slash -- tokenizer is sensitive to it
    ids, mask = _tokenize(url)
    logits = sess.run(None, {
        IN0: np.array([ids],  dtype=np.int64),
        IN1: np.array([mask], dtype=np.int64),
    })[0][0]
    eb, ep = math.exp(logits[0]), math.exp(logits[1])
    return ep / (eb + ep)

# Smoke test
print(f'Smoke test  -- google.com p_phishing={bert_score("https://www.google.com"):.4f}  (expect < 0.20)')

Vocab loaded   (400 tokens)
URLBert loaded (models/urlbert.onnx)
Smoke test  -- google.com p_phishing=0.0653  (expect < 0.20)


In [6]:
# ── Cell 4: Real TFLite CNN + Fusion engine + Context adjuster ─────
import re as _re_fusion

# ── Load real TFLite CNN ──────────────────────────────────────────
TFLITE_MODEL    = 'models/safelink_model.tflite'
MAX_SEQ_LEN     = 200
CNN_TEMPERATURE = 5.0

_cnn_runner = None
if TFLiteInterpreter is not None and os.path.exists(TFLITE_MODEL):
    _tflite_interp = TFLiteInterpreter(model_path=TFLITE_MODEL)
    _cnn_runner    = _tflite_interp.get_signature_runner('serving_default')
    _cnn_output_key = list(_cnn_runner.get_output_details().keys())[0]
    print(f'CNN model  : loaded ({TFLITE_MODEL})  [{_TFLITE_BACKEND}]')
    print(f'  inputs   : {list(_cnn_runner.get_input_details().keys())}')
    print(f'  outputs  : {list(_cnn_runner.get_output_details().keys())}')
else:
    _reason = 'model file missing' if TFLiteInterpreter is not None else 'ai-edge-litert not installed'
    print(f'CNN model  : heuristic mock ({_reason})')

def _url_tokenize(url):
    ids = np.zeros(MAX_SEQ_LEN, dtype=np.int32)
    for i, ch in enumerate(url[:MAX_SEQ_LEN]):
        ids[i] = max(1, min(99, ord(ch) - 31))
    return ids

def _temperature_scale(p_mal_raw, T=CNN_TEMPERATURE):
    eps = 1e-6
    p   = max(eps, min(1 - eps, p_mal_raw))
    logit = math.log(p / (1 - p))
    return 1 / (1 + math.exp(-logit / T))

def _real_cnn(url, scaled):
    seq  = _url_tokenize(url).reshape(1, MAX_SEQ_LEN)
    feat = scaled.reshape(1, -1).astype(np.float32)
    out  = _cnn_runner(seq_input=seq, num_input=feat)
    probs      = out[_cnn_output_key][0]
    p_mal_cal  = _temperature_scale(float(probs[2]))
    p_safe_cal = 1.0 - p_mal_cal
    return p_safe_cal, float(probs[1]), p_mal_cal

def _mock_cnn(scaled):
    idx = FEATURE_COLUMNS.index
    score = float(np.clip(
        scaled[idx('has_suspicious_tld')]     * 0.45 +
        scaled[idx('phishing_keyword_count')] * 0.40 +
        scaled[idx('num_at_signs')]           * 0.35 +
        scaled[idx('is_ip_address')]          * 0.30 +
        scaled[idx('num_hyphens')]            * 0.18 +
        scaled[idx('has_url_shortener')]      * 0.15 +
        scaled[idx('has_hex_encoding')]       * 0.12 +
        scaled[idx('is_https')]               * -0.10 +
        scaled[idx('has_trusted_tld')]        * -0.10,
        -3, 3
    ))
    p_mal = 1 / (1 + math.exp(-score))
    return 1.0 - p_mal, 0.0, p_mal

def _cnn(url, scaled):
    if _cnn_runner is not None:
        return _real_cnn(url, scaled)
    return _mock_cnn(scaled)

def _mock_autoencoder(_scaled):
    return 0.005

# ── Fusion engine (mirrors FusionEngine.kt) ───────────────────────
CNN_HIGH_CONF           = 0.85
CNN_BERT_OVERRIDE_LIMIT = 0.88  # above this CNN is too confident to be overridden by URLBert
DIVERGENCE_THR          = 0.40
ANOMALY_THR             = 0.02
BLEND_THR               = 0.55  # Rule 5 only — raised from 0.50 to reduce FP on borderline blends
W_CNN, W_BERT           = 0.70, 0.30  # raised CNN weight to reduce BERT-driven FPs
BERT_CONFIDENT_BENIGN   = 0.12
BERT_CONFIDENT_PHISHING = 0.88


# ── Option A/E helpers: structural phishing evidence ─────────────────────
# Indices of features that represent concrete phishing structure.
# If ALL are zero, both models are firing on URL character patterns alone.
_PHISHING_FEAT_IDX = [8, 18, 20, 22, 23, 24, 32, 34]  # at_sign, ip, susp_tld, phish_kw, brand_kw, shortener, hex, subdomain_trusted

# Option E: minimum weighted structural-risk score required to sustain MALICIOUS.
STRUCTURAL_RISK_MIN = 1.0
_STRUCT_RISK_WEIGHTS = {8: 5.0, 18: 4.0, 20: 4.5, 22: 4.0, 23: 3.8, 24: 3.2, 32: 1.3, 34: 3.5}

def _brand_kw_structural(raw_vec):
    """Count brand_keyword as structural risk only when paired with suspicious host patterns."""
    if len(raw_vec) <= 23 or raw_vec[23] <= 0:
        return False
    has_hyphens        = len(raw_vec) > 5  and raw_vec[5]  > 0
    has_suspicious_tld = len(raw_vec) > 20 and raw_vec[20] > 0
    has_phish_kw       = len(raw_vec) > 22 and raw_vec[22] > 0
    has_special_chars  = len(raw_vec) > 30 and raw_vec[30] > 0
    has_digit_run      = len(raw_vec) > 33 and raw_vec[33] > 0
    deep_subdomain     = len(raw_vec) > 19 and raw_vec[19] >= 2
    return any((has_hyphens, has_suspicious_tld, has_phish_kw, has_special_chars, has_digit_run, deep_subdomain))

def _all_phishing_clean(raw_vec):
    """Option A: True when every critical phishing feature is zero.
    Means models are firing on character/token patterns alone -- no structural evidence."""
    has_hyphens        = len(raw_vec) > 5  and raw_vec[5]  > 0
    has_suspicious_tld = len(raw_vec) > 20 and raw_vec[20] > 0
    def _kw_clean(i):
        if i == 22:
            return not has_hyphens and not has_suspicious_tld
        if i == 23:
            return not _brand_kw_structural(raw_vec)
        return False
    return all(raw_vec[i] == 0 or _kw_clean(i) for i in _PHISHING_FEAT_IDX if i < len(raw_vec))

def _structural_risk(raw_vec):
    """Option E: Weighted sum of phishing-structural feature values.
    Returns 0.0 when there is zero structural evidence of phishing."""
    has_hyphens        = len(raw_vec) > 5  and raw_vec[5]  > 0
    has_suspicious_tld = len(raw_vec) > 20 and raw_vec[20] > 0
    return sum(w for i, w in _STRUCT_RISK_WEIGHTS.items()
               if i < len(raw_vec) and raw_vec[i] > 0
               and not (i == 22 and not has_hyphens and not has_suspicious_tld)
               and not (i == 23 and not _brand_kw_structural(raw_vec)))

def _fuse(p_safe, _pw, p_mal, bert, anomaly_mse, raw_vec=None):
    """Fusion engine (mirrors FusionEngine.kt).
    raw_vec: raw feature vector -- enables Option A feature-nullity gate."""
    cnn_conf   = max(p_safe, p_mal)
    is_anomaly = anomaly_mse > ANOMALY_THR

    # --- Rule 0: Hard structural pre-rule -- hyphen-stuffed brand impersonation ---
    # Mirrors FusionEngine.kt Rule 0: host with >=2 hyphens + >=2 phishing keywords
    # + >=1 brand keyword is a textbook phishing domain. No model score can override.
    if raw_vec is not None:
        _num_hyphens = raw_vec[5]  if len(raw_vec) > 5  else 0
        _phish_kw    = raw_vec[22] if len(raw_vec) > 22 else 0
        _brand_kw    = raw_vec[23] if len(raw_vec) > 23 else 0
        if _num_hyphens >= 2 and _phish_kw >= 2 and _brand_kw >= 1:
            return ('MALICIOUS', 0.97,
                    f'Rule 0 (hyphen-stuffed brand impersonation: '
                    f'hyphens={int(_num_hyphens)}, '
                    f'phishKw={int(_phish_kw)}, '
                    f'brandKw={int(_brand_kw)})')

    if cnn_conf > CNN_HIGH_CONF:
        if p_mal > p_safe:
            if bert < BERT_CONFIDENT_BENIGN:
                # URLBert override: only allowed when CNN is not too confident.
                # Above CNN_BERT_OVERRIDE_LIMIT (0.88) the CNN signal is strong enough
                # that URLBert should not dismiss it.
                if p_mal < CNN_BERT_OVERRIDE_LIMIT:
                    return 'SAFE', 1.0 - bert, f'Rule 1 (URLBert overrides CNN -- bert={bert:.3f})'
                # CNN too confident to be overridden -- fall through to MALICIOUS
            elif bert < 0.50:
                # Option A-extended: CNN fires high but BERT leans safe.
                # If ALL structural phishing features are zero, CNN is reacting to URL
                # character patterns only (e.g. #/login in a SPA route) -- return SAFE.
                if raw_vec is not None and _all_phishing_clean(raw_vec):
                    return 'SAFE', max(0.5, 1.0 - bert), f'Rule 1 (CNN pattern-only, BERT safe -- bert={bert:.3f}, no structural evidence)'
                return 'WARNING', 0.65, f'Rule 1 (CNN-URLBert disagree: bert={bert:.3f})'
            # ── Option A: Feature-Nullity Gate ────────────────────────────
            # CNN > CNN_HIGH_CONF AND URLBert also agrees (both > 0.5),
            # but if ALL critical phishing features are zero the models fired
            # on URL character/token patterns alone (e.g. repeated segment in
            # path, product numbers like -15-5g).  Assert WARNING, not MALICIOUS.
            # Exception: both models very confident (bert>0.88) — typosquatting bypasses structural gate
            if raw_vec is not None and _all_phishing_clean(raw_vec) and bert <= BERT_CONFIDENT_PHISHING:
                return 'WARNING', 0.65, 'Rule 1b (pattern-only: no structural phishing evidence)'
            # ──────────────────────────────────────────────────────────────
        v = 'MALICIOUS' if p_mal > p_safe else 'SAFE'
        return v, p_mal if v == 'MALICIOUS' else p_safe, 'Rule 1 (CNN high-conf)'
    div            = abs(p_mal - bert)
    bert_uncertain = BERT_CONFIDENT_BENIGN < bert < BERT_CONFIDENT_PHISHING
    cnn_override   = p_mal > 0.60 and bert < BERT_CONFIDENT_BENIGN and div > 0.55
    # Rule 2 guard: skip WARNING if CNN is low-conf AND no structural features
    # (prevents Rule 2 WARNING on clean domains where URLBert is merely uncertain)
    _r2_should_warn = (raw_vec is None or not _all_phishing_clean(raw_vec) or p_mal >= 0.55)
    if div > DIVERGENCE_THR and (bert_uncertain or cnn_override) and _r2_should_warn:
        return 'WARNING', 0.65, f'Rule 2 (divergence={div:.2f}, CNN={p_mal:.3f}, BERT={bert:.3f})'
    if is_anomaly and cnn_conf < 0.60:
        return 'WARNING', 0.60, 'Rule 3 (anomaly)'
    if p_mal > 0.5 and bert > 0.5:
        return 'MALICIOUS', max(p_mal, bert), 'Rule 4 (both agree: phishing)'
    if p_safe > 0.5 and bert < 0.5:
        if raw_vec is not None and _structural_risk(raw_vec) >= 3.5:
            return 'WARNING', 0.70, f'Rule 4s (models agree safe but structural_risk={_structural_risk(raw_vec):.1f})'
        return 'SAFE', max(p_safe, 1 - bert), 'Rule 4 (both agree: safe)'
    blend = p_mal * W_CNN + bert * W_BERT
    v = 'MALICIOUS' if blend > BLEND_THR else 'SAFE'
    if v == 'SAFE' and raw_vec is not None and _structural_risk(raw_vec) >= 3.5:
        return 'WARNING', 0.70, f'Rule 5s (blend safe but structural_risk={_structural_risk(raw_vec):.1f})'
    return v, blend if v == 'MALICIOUS' else 1 - blend, f'Rule 5 (blend={blend:.2f})'

# ── Context adjuster (mirrors SafeLinkPredictor.contextAdjust()) ──
# -- Context adjuster (mirrors SafeLinkPredictor.contextAdjust()) --
_USER_HOSTED_PLATFORMS = {
    'wixsite.com', 'weebly.com', 'webflow.io', 'wordpress.com',
    'blogspot.com', 'squarespace.com', 'sites.google.com',
    'github.io', 'netlify.app', 'vercel.app', 'cargo.site',
}
_SAFE_PATH_WORDS = {
    'portfolio', 'about', 'blog', 'resume', 'cv', 'gallery',
    'contact', 'projects', 'work', 'services', 'showcase',
}

def _context_adjust(url, raw_vec, verdict, conf, rule):
    # User-hosting platform subdomains: the registered domain (wixsite.com etc.) is always
    try:
        _h = _re_fusion.sub(r'^https?://', '', url.strip()).split('/')[0].split(':')[0].lower()
        _h = _re_fusion.sub(r'^www\.', '', _h)
    except Exception:
        _h = ''
    _platform = next((p for p in _USER_HOSTED_PLATFORMS if _h != p and _h.endswith('.' + p)), None)
    if _platform and verdict in ('SAFE', 'WARNING'):
        return 'WARNING', max(conf, 0.60), f'User-hosting platform ({_platform}) — anyone can publish here'
    if verdict != 'MALICIOUS':
        return verdict, conf, rule
    try:
        host = _re_fusion.sub(r'^https?://', '', url.strip()).split('/')[0].split(':')[0].lower()
        host = _re_fusion.sub(r'^www\.', '', host)
        path = (_re_fusion.sub(r'^https?://[^/]+', '', url)).lower()
    except Exception:
        return verdict, conf, rule
    is_hosted    = any(host == p or host.endswith('.' + p) for p in _USER_HOSTED_PLATFORMS)
    phishing_kw  = raw_vec[22] if len(raw_vec) > 22 else 0
    path_words   = set(_re_fusion.split(r'[/\-_\.\?&=]', path))
    has_safe_word = phishing_kw == 0 and bool(_SAFE_PATH_WORDS & path_words)
    if not is_hosted and not has_safe_word:
        return verdict, conf, rule
    tags = []
    if is_hosted:     tags.append('user-hosting platform')
    if has_safe_word: tags.append('safe path word')
    return 'WARNING', 0.60, f'{rule} -> capped WARNING ({", ".join(tags)})'


# ── Confidence gate: MALICIOUS only for blocklist or conf ≥90% ───────────────
_MALICIOUS_CONF_MIN = 0.90

def _apply_confidence_gate(verdict, conf, rule):
    """MALICIOUS is reserved for L0-blocklist hits and very high-confidence
    predictions (conf >= 90%). Below that threshold the verdict is downgraded
    to WARNING so users are warned but not over-alarmed."""
    if verdict == 'MALICIOUS' and conf < _MALICIOUS_CONF_MIN:
        return 'WARNING', conf, rule
    return verdict, conf, rule

print('Fusion + context adjuster ready.')

CNN model  : loaded (models/safelink_model.tflite)  [ai_edge_litert]
  inputs   : ['num_input', 'seq_input']
  outputs  : ['output_0']
Fusion + context adjuster ready.


In [7]:
# ── Cell 5: L0/L1 guards + XAI engine (mirrors Android pipeline) ──
import os as _os

# ── L1 trusted TLDs (mirrors SafeLinkPredictor.TRUSTED_TLDS) ──────
_TRUSTED_TLDS_L1 = {'gov', 'edu', 'mil', 'ac'}

def _tld_of(url):
    """Return (full_host_no_www, tld) from a URL."""
    try:
        host = _re.sub(r'^https?://', '', url.strip()).split('/')[0].split(':')[0].lower()
        host = _re.sub(r'^www\.', '', host)
        tld  = host.split('.')[-1] if '.' in host else ''
        return host, tld
    except Exception:
        return '', ''

def _is_trusted_tld(url):
    host, tld = _tld_of(url)
    return (tld in _TRUSTED_TLDS_L1 or
            host.endswith('.gov.lk') or
            host.endswith('.ac.lk')  or
            host.endswith('.edu.lk'))

# ── L0 blocklist (load from Android assets if path is reachable) ──
_BLOCKLIST = set()
_BL_PATH = r'D:\AndroidStudio Projects\SafeLink Latest\app\src\main\assets\blocklist.txt'
if _os.path.exists(_BL_PATH):
    with open(_BL_PATH, encoding='utf-8') as _f:
        _BLOCKLIST = {ln.strip().lower() for ln in _f if ln.strip() and not ln.startswith('#')}
    print(f'Blocklist loaded  : {len(_BLOCKLIST)} entries')
else:
    print('Blocklist         : not found -- L0-BLOCKLIST check skipped')

# ── XAI engine (mirrors XAIEngine.kt exactly) ─────────────────────
# (feat_idx, threshold, weight, reason, invert)
# Fires when: invert=False -> raw[idx] > threshold
#             invert=True  -> raw[idx] <= threshold  (e.g. is_https=0 -> not secure)
_XAI_RULES = [
    (8,  0,   5.0, "This link contains an '@' sign -- a trick used to hide the real website address",                    False),
    (20, 0,   4.5, "This link has a suspicious web address ending (like .xyz or .tk) that is commonly used for scams",  False),
    (22, 0,   4.0, "This link contains suspicious words like 'login', 'verify', or 'secure' -- banks never ask for passwords via links", False),
    (23, 0,   3.8, "This link pretends to be a well-known brand or bank to trick you",                                  False),
    (18, 0,   3.5, "This link goes directly to an IP address instead of a real website name -- very unusual and suspicious", False),
    (24, 0,   3.2, "This is a shortened link that hides the real destination",                                          False),
    (0,  80,  2.8, "This link is unusually long -- scammers use long links to confuse people",                           False),
    (4,  4,   2.5, "This link has too many dots, which is a common trick used in phishing",                             False),
    (5,  3,   2.3, "This link has many hyphens, which scammers use to make fake sites look real",                       False),
    (34, 0,   2.0, "This link has a real website address hidden inside a longer fake one",                              False),
    (17, 0,   1.8, "This link does not use a secure connection (no padlock) -- your information could be stolen",        True),
    (33, 2,   1.5, "This link contains lots of numbers in the website address, which is unusual for real websites",     False),
    (32, 0,   1.3, "This link uses special encoded characters to hide its true destination",                            False),
    (26, 0,   1.2, "This link uses an unusual port number, which is not normal for real websites",                      False),
    (27, 0,   1.0, "This link has an unusual double-slash in the path, which may indicate a redirect trick",            False),
    (19, 2,   0.9, "This link has many sub-addresses, which can be used to disguise a fake website",                   False),
]

def _xai_explain(raw_vec, verdict):
    """Mirrors XAIEngine.explain() -- returns (primary_reason, all_reasons_list)."""
    fired = []
    for feat_idx, threshold, weight, reason, invert in _XAI_RULES:
        if feat_idx >= len(raw_vec):
            continue
        value = raw_vec[feat_idx]
        triggered = (value <= threshold) if invert else (value > threshold)
        if triggered:
            fired.append((weight, reason))
    fired.sort(key=lambda x: -x[0])
    top3 = fired[:3]
    if top3:
        return top3[0][1], [r[1] for r in top3]
    fallback = {
        'MALICIOUS': "This link shows multiple signs of a phishing or malware attack",
        'WARNING':   "This link has some suspicious features -- proceed with caution",
    }.get(verdict, "This link appears to be safe")
    return fallback, [fallback]

print('XAI engine ready.')

Blocklist         : not found -- L0-BLOCKLIST check skipped
XAI engine ready.


In [8]:
# ── Cell 6: check_url() — full pipeline ───────────────────────────
import re as _re

_TRUSTED_ROOTS = {
    'adaderana.lk','adobe.com','airtel.lk','akamaied.net','amazon.com','anthropic.com','apple.com','claude.ai','atlassian.com',
    'bing.com','blogspot.com','boc.lk','booking.com','canva.com','cargillsce.com','cbsl.gov.lk','ceb.lk',
    'cloudflare-dns.com','cloudflare.com','cmb.ac.lk','combank.lk','cursor.com','customs.gov.lk','dailymirror.lk','daraz.lk',
    'dfcc.lk','dialog.lk','docs.google.com','doenets.lk','dominos.lk','dpeducation.org','drive.google.com','dropbox.com',
    'duckduckgo.com','ebay.com','elections.gov.lk','esn.ac.lk','ezcash.lk','facebook.com','fastly.net','forms.gle',
    'frimi.lk','gazette.lk','gemini.google.com','genie.lk','github.com','glomark.lk','gmail.com','google.com','google.lk',
    'googleapis.com','googlevideo.com','gov.lk','grasshoppers.lk','gstatic.com','hattonbank.lk','haveibeenpwned.com','health.gov.lk',
    'hirunews.lk','hnb.lk','hutch.lk','iana.org','ikman.lk','immigration.gov.lk','instagram.com','ipay.lk','ird.gov.lk',
    'island.lk','jsdelivr.net','kapruka.com','keellssuper.com','kln.ac.lk','lankagate.gov.lk','lassana.com','letsencrypt.org',
    'linear.app','linkedin.com','live.com','mcash.lk','medium.com','microsoft.com','mobitel.lk','moe.gov.lk','motortraffic.gov.lk',
    'ndbbank.lk','netflix.com','newsfirst.lk','nibm.lk','notion.so','nsblank.lk','office.com','openai.com','ou.ac.lk',
    'outlook.com','panasian.lk','payhere.lk','paypal.com','pdn.ac.lk','peoples.lk','perplexity.ai','pickme.lk','pizzahut.lk',
    'police.gov.lk','protonmail.com','reddit.com','ruh.ac.lk','sampath.lk','schema.org','seylanbank.lk','sjp.ac.lk',
    'slack.com','slpost.gov.lk','slt.lk','spotify.com','srilankan.com','stackoverflow.com','stripe.com','sundaytimes.lk',
    'teams.microsoft.com','tiktok.com','trello.com','twitch.tv','twitter.com','uber.com','unionb.lk','university.lk','unpkg.com',
    'uom.lk','virustotal.com','w3.org','waterboard.lk','whatsapp.com','wikipedia.org','wordpress.com','x.com','yahoo.com',
    'youtu.be','youtube.com','zoom.us',
    # ── added to fix FPs ──
    '99designs.com',
    '9to5mac.com',
    'abn-amro.nl',
    'adobe.com.co',
    'adorama.com',
    'alibaba.com',
    'aljazeera.com',
    'allrecipes.com',
    'amazon.ca',
    'amazon.co.jp',
    'amazon.com.cl',
    'amazon.com.co',
    'amazon.fr',
    'americanexpress.com',
    'anandtech.com',
    'androidauthority.com',
    'ansible.com',
    'apache.org',
    'apnews.com',
    'apple.co.jp',
    'apple.com.cl',
    'apple.com.co',
    'appleinsider.com',
    'arstechnica.com',
    'audiosciencereview.com',
    'autodesk.com',
    'autotrader.com',
    'avsforum.com',
    'baidu.com',
    'bandcamp.com',
    'bankofamerica.com',
    'betanews.com',
    'bhphotovideo.com',
    'blizzard.com',
    'bloomberg.com',
    'bluehost.com',
    'bnpparibas.com',
    'bonappetit.com',
    'bootstrapcdn.com',
    'britishairways.com',
    'buymeacoffee.com',
    'buzzfeed.com',
    'capitalone.com',
    'carfax.com',
    'cashapp.com',
    'cbsnews.com',
    'cisco.com',
    'coinbase.com',
    'costco.com',
    'creativemarket.com',
    'creditsuisse.com',
    'crunchyroll.com',
    'cultofmac.com',
    'deliveroo.com',
    'deutsche-bank.de',
    'digitalocean.com',
    'digitaltrends.com',
    'doordash.com',
    'dreamhost.com',
    'dropbox.com.cl',
    'dropbox.com.co',
    'duolingo.com',
    'ebay.co.jp',
    'ebay.com.co',
    'epicurious.com',
    'euronews.com',
    'everydayhealth.com',
    'facebook.com.co',
    'fastmail.com',
    'fontawesome.com',
    'france24.com',
    'google.com.co',
    'grubhub.com',
    'hbomax.com',
    'hdtvtest.com',
    'huffpost.com',
    'humblebundle.com',
    'hyatt.com',
    'ibm.com',
    'influxdata.com',
    'instagram.co.jp',
    'instagram.in',
    'instagram.pl',
    'intel.com',
    'invisionapp.com',
    'irishexaminer.com',
    'jira.com',
    'jquery.com',
    'jsfiddle.net',
    'justeat.com',
    'kickstarter.com',
    'ko-fi.com',
    'kotaku.com',
    'kraken.com',
    'kyoto-u.ac.jp',
    'linkedin.com.co',
    'lucidchart.com',
    'lufthansa.com',
    'macrumors.com',
    'mailchimp.com',
    'makeuseof.com',
    'mastercard.com',
    'medicinenet.com',
    'metacritic.com',
    'microsoft.co.jp',
    'microsoft.com.co',
    'mozilla.org',
    'mysql.com',
    'namecheap.com',
    'natwest.com',
    'nbcnews.com',
    'neo4j.com',
    'neowin.com',
    'netflix.co.jp',
    'newegg.com',
    'newyorker.com',
    'nintendo.com',
    'notebookcheck.com',
    'npmjs.com',
    'nvidia.com',
    'nytimes.com',
    'onedrive.com',
    'opentable.com',
    'oracle.com',
    'orbitz.com',
    'panasonic.com',
    'paramountplus.com',
    'paypal.com.cl',
    'paypal.com.co',
    'peacocktv.com',
    'phonearena.com',
    'pixabay.com',
    'pluralsight.com',
    'qatarairways.com',
    'quora.com',
    'railway.app',
    'rfc-editor.org',
    'robinhood.com',
    'rtings.com',
    'rxlist.com',
    'salesforce.com',
    'samsung.com',
    'santander.com',
    'schwab.com',
    'scotsman.com',
    'seamless.com',
    'seriouseats.com',
    'sharepoint.com',
    'shopify.com',
    'singaporeair.com',
    'skillshare.com',
    'skyscanner.com',
    'snapchat.com',
    'sony.com',
    'soundonsound.com',
    'southwest.com',
    'springer.com',
    'sputniknews.com',
    'steampowered.com',
    'stereonet.com',
    'substack.com',
    'supabase.com',
    'tdameritrade.com',
    'thurrott.com',
    'tomshardware.com',
    'toshiba.com',
    'travis-ci.com',
    'tumblr.com',
    'tutanota.com',
    'ubisoft.com',
    'unsplash.com',
    'uptodate.com',
    'usatoday.com',
    'vanguard.com',
    'venturebeat.com',
    'vmware.com',
    'vultr.com',
    'windowscentral.com',
    'wyndhamhotels.com',
    'xbox.com',
    'yandex.com',
    'zappos.com',
    'zillow.com',
    'ziprecruiter.com',
    'zomato.com',
    'zzounds.com',
}

def _root_domain(url):
    try:
        host  = _re.sub(r'^https?://', '', url.strip()).split('/')[0].split(':')[0].lower()
        parts = host.split('.')
        if len(parts) >= 3 and parts[-2] in ('co','gov','org','net','ac','edu','com'):
            return '.'.join(parts[-3:])
        if len(parts) >= 2:
            return '.'.join(parts[-2:])
    except Exception:
        pass
    return ''

def _host_of(url):
    try:
        h = _re.sub(r'^https?://', '', url.strip()).split('/')[0].split(':')[0].lower()
        return _re.sub(r'^www\.', '', h)
    except Exception:
        return ''

def _whitelisted(url):
    return _root_domain(url) in _TRUSTED_ROOTS

def _normalize(url):
    url = url.strip()
    if url and not _re.match(r'^https?://', url, _re.IGNORECASE):
        url = 'https://' + url
    return url

BADGE      = {'SAFE': '✅ SAFE', 'WARNING': '⚠️  WARNING', 'MALICIOUS': '🚨 MALICIOUS', 'BLOCKED': '🛑 BLOCKED'}
_CNN_LABEL = 'real TFLite' if _cnn_runner is not None else 'heuristic mock'

def check_url(raw_input):
    if not raw_input or not raw_input.strip():
        print('Provide a URL.'); return

    url      = _normalize(raw_input)
    original = raw_input.strip()
    norm_tag = f'  [normalized: {original}]' if url != original else ''

    t0   = time.perf_counter()
    host = _host_of(url)

    # ── L0: blocklist ─────────────────────────────────────────────
    if host and host in _BLOCKLIST:
        ms = (time.perf_counter() - t0) * 1000
        print()
        print(f'  {BADGE["BLOCKED"]}  (100.0% confidence)')
        print(f'  URL    : {url}{norm_tag}')
        print(f'  Layer  : L0-BLOCKLIST')
        print(f'  Reason : Domain is on the phishing/malware blocklist')
        print(f'  Time   : {ms:.1f} ms')
        print()
        return

    # ── L1: static whitelist ──────────────────────────────────────
    if _whitelisted(url):
        ms = (time.perf_counter() - t0) * 1000
        print()
        print(f'  {BADGE["SAFE"]}  (100.0% confidence)')
        print(f'  URL    : {url}{norm_tag}')
        print(f'  Layer  : L1-WHITELIST  [{_root_domain(url)}]')
        print(f'  Time   : {ms:.1f} ms')
        print()
        return

    # ── L1: trusted TLD ───────────────────────────────────────────
    if _is_trusted_tld(url):
        ms = (time.perf_counter() - t0) * 1000
        print()
        print(f'  {BADGE["SAFE"]}  (100.0% confidence)')
        print(f'  URL    : {url}{norm_tag}')
        print(f'  Layer  : L1-TRUSTED_TLD')
        print(f'  Time   : {ms:.1f} ms')
        print()
        return

    # ── L2: ML hybrid ─────────────────────────────────────────────
    raw                   = extract_feature_vector(url)
    scaled                = scale_features(raw)
    p_safe, p_warn, p_mal = _cnn(url, scaled)
    anomaly               = _mock_autoencoder(scaled)
    bert                  = bert_score(url)
    verdict, conf, rule   = _fuse(p_safe, p_warn, p_mal, bert, anomaly, raw)
    verdict, conf, rule   = _context_adjust(url, raw, verdict, conf, rule)
    verdict, conf, rule   = _apply_confidence_gate(verdict, conf, rule)
    primary_reason, all_reasons = _xai_explain(raw, verdict)


    layer      = 'L2-ML'
    ms = (time.perf_counter() - t0) * 1000
    print()
    print(f'  {BADGE[verdict]}  ({conf*100:.1f}% confidence)')
    print(f'  URL        : {url}{norm_tag}')
    print(f'  Layer      : {layer}')
    print(f'  Rule       : {rule}')
    print(f'  Reason     : {primary_reason}')
    for r in all_reasons[1:]:
        print(f'             + {r}')
    print(f'  CNN        : safe={p_safe:.3f}  malicious={p_mal:.3f}  ({_CNN_LABEL})')
    print(f'  URLBert    : p_phishing={bert:.4f}')

    active = [(k, raw[FEATURE_COLUMNS.index(k)])
              for k in ['is_https','has_suspicious_tld','phishing_keyword_count',
                        'num_at_signs','has_url_shortener','num_hyphens',
                        'is_ip_address','brand_keyword_count','has_hex_encoding',
                        'has_port_in_url','has_tld_in_path']
              if raw[FEATURE_COLUMNS.index(k)] != 0]
    if active:
        print(f'  Features   : ' + '  |  '.join(f'{k}={v}' for k, v in active))
    print(f'  Time       : {ms:.1f} ms')
    print()

print('check_url() ready — run the single-URL cell below.')

check_url() ready — run the single-URL cell below.


In [9]:
# ── Cell 8: Batch test — compact table view ───────────────────────
# Flags — toggle these before running

TEST_URLS = [
    'https://www.google.com',
    'https://hnb.lk/personal/internet-banking',
    'https://www.dialog.lk/en/mobile-broadband',
    'http://paypal-secure-login.xyz/verify',
    'https://nawferadnan15.wixsite.com/ad-portfolio',
    'https://learn.zoom.us/j/97129300588?pwd=7EertfewlCrSZaNlavG5L73Lt77Har.1',
    'http://192.168.1.1/admin/phishing-page',
    'http://evil.xyz/www.paypal.com/account/login',
    'https://www.dialog.lk/',
    'https://www.dialog.lk/buy/new-connection/hbb',
    'google.com/security-login-update',
    'https://claude.com/contact-sales/claude-for-oss?utm_source=sp_auto_dm',
    'https://github.com/affaan-m/everything-claude-code',
    'https://mastershuhad.github.io/EasyAssist/',
    'https://www.menti.com/alri4n233us6?source=qr-page',
    'https://paypal.com.verify-account.ru',
    'https://laka.djkla.xyz/el6japmj/1588c2470a6fe0163bfe940283?_t=1761660405048&p=w',
    'https://001myacct.weebly.com/',
    'https://pixabay.com/',
    'laka.djkla.xyz/el6japmj/1588c2470a6fe0163bfe940283',
    'https://mpmp.wwtic.xyz/w3vjtqlx/6b96fb8dee0993bb9020a4a24b?_t=1770045924022&p=w',
    'https://lihi1.me/eRImR',
    'https://amazon-delivery-issue-secure.com/confirm-shipment',
    'https://microsoft365-verify-account.net/login-secure',
    'https://appleid-icloud-security-update.com/verify-device',
    'https://paypal-account-confirmation-urgent.live/secure-login',
    'https://accounts-google-security-alert.com/reset-password',
    'https://irs-gov-tax-refund-claim.com/verify-identity',
    'https://free-bitcoin-airdrop-bonus.net/claim-now',
    'https://fedex-tracking-update-secure.com/package/confirm',
    'https://netflix-account-verification.com/renew-subscription',
    'https://zoom-meeting-invite-urgent.net/join-secure',
    'https://paypa1-secure.com/login',
    'https://arnazon-order-confirm.co',
    'https://g00gle-accounts.com/verify',
    'https://micr0soft-support.net',
    'https://app1e-id.com/icloud',
    'https://faceb00k-security-alert.com',
    'https://amaz0n-prime-delivery.net',
    'https://secure-login-paypal-verification.com/account',
    'https://update-microsoft365.com/renew-license',
    'https://amazon-package-delay-claim.com/track',
    'https://support-apple.com-verify.com',
    'https://crypto-elon-giveaway.live/claim-reward',
    'https://bankofamerica-login-secure.net',
    'https://whatsapp-verification-update.com',
    ' http://volvogroup.com/',
   'https://www.thyssenkrupp.com/',
   'http://tutech.net/index.php/page/about-us-2011-03-04#1',
  ' http://www.gabo-mi.com/',
   '   http://www.sirehna.com/',
    ' https://www.inesc-id.pt/',
  '  http://um.edu.ml/',
    '  http://www.hadassah-med.com/',
     ' http://new.tu-varna.bg/en/index.php/bg/',
       '  http://www.iunet.info/',
       ' https://www.global.hokudai.ac.jp/',
     ' https://maichindom.com/',
      'https://www.hsu-hh.de/hsu/index.php',
       'http://web.ncku.edu.tw/bin/home.php',
  'zzgylc.com',
'zzht001.cn',
'zzxzsj.com',
'zzxzzx666.com',
'zzzzzzbxxxx.weebly.com',
'zydsmkifgi.duckdns.org',
'zyj.spencerleedslaw.com'
'zylen-hold.pages.dev',
'zylon-hub.pages.dev',
'zymsvdz.cn',
'zymylyydkw.duckdns.org',
'zynibvruu75.click',
'zyra-fount.pages.dev',
'zyra-studio.pages.dev',
'zytrezorlogeu.webflow.io',
'zyzxfxyq.com',
'zzarhxmjlu.blogspot.com',
'zzarhxmjlu.blogspot.tw',
'zzatztc.cn',
'zzb.vrncmwsk.es',
'zz.empfehlenswert.cf'
'100.25.1.9',
'100.42.65.208',
'100.42.65.230',
'101.0.81.153',
'101.132.78.77',
'https://aisuok.dub.link/blog',
'https://aisuok.dub.link/tiktok',
'https://aisuok.dub.link/youtube',
'https://aisuok.dub.link/whatsapp',
'https://aisuok.dub.link/linkedin',
'https://aisuok.dub.link/facebook',
'https://aisuok.dub.link/instagram',


]

# ── Internal scanner ──────────────────────────────────────────────
def _scan(raw_input):
    url  = _normalize(raw_input)
    host = _host_of(url)

    if host and host in _BLOCKLIST:
        return dict(verdict='BLOCKED', conf=1.0, cnn_mal=None, bert=None,
                    rule='L0-blocklist', url=url)
    if _whitelisted(url):
        return dict(verdict='SAFE', conf=1.0, cnn_mal=None, bert=None,
                    rule='L1-whitelist', url=url)
    if _is_trusted_tld(url):
        return dict(verdict='SAFE', conf=1.0, cnn_mal=None, bert=None,
                    rule='L1-trusted-tld', url=url)

    raw                   = extract_feature_vector(url)
    scaled                = scale_features(raw)
    p_safe, p_warn, p_mal = _cnn(url, scaled)
    anomaly               = _mock_autoencoder(scaled)
    bert                  = bert_score(url)
    verdict, conf, rule   = _fuse(p_safe, p_warn, p_mal, bert, anomaly, raw)
    verdict, conf, rule   = _context_adjust(url, raw, verdict, conf, rule)
    verdict, conf, rule   = _apply_confidence_gate(verdict, conf, rule)

    return dict(verdict=verdict, conf=conf, cnn_mal=p_mal, bert=bert,
                rule=rule, url=url)

# ── Table renderer ─────────────────────────────────────────────────
_V_ICON  = {'SAFE': '✅', 'WARNING': '⚠️ ', 'MALICIOUS': '🚨', 'BLOCKED': '🛑'}
_V_SHORT = {'SAFE': 'SAFE     ', 'WARNING': 'WARNING  ', 'MALICIOUS': 'MALICIOUS', 'BLOCKED': 'BLOCKED  '}

def _fmt_score(v):
    return f'{v:.3f}' if v is not None else '  -  '


HDR = f"{'#':>3}  {'VERDICT':<10} {'CONF':>6}  {'CNN':>5}  {'BERT':>5} {'G':>2}  {'RULE':<32}  URL"
SEP = '-' * 135

print(HDR)
print(SEP)
for i, raw_in in enumerate(TEST_URLS, 1):
    r      = _scan(raw_in)
    icon   = _V_ICON.get(r['verdict'], '?')
    vshort = _V_SHORT.get(r['verdict'], r['verdict'])
    conf   = f"{r['conf']*100:5.1f}%"
    cnn    = _fmt_score(r['cnn_mal'])
    bert   = _fmt_score(r['bert'])
    rule   = r['rule'][:32]
    url    = r['url'][:65]
    print(f"{i:>3}  {icon} {vshort} {conf}  {cnn}  {bert}  {rule:<32}  {url}")

print(SEP)
counts = {v: sum(1 for r in [_scan(u) for u in TEST_URLS] if r['verdict'] == v) for v in ['SAFE','WARNING','MALICIOUS','BLOCKED']}
print(f"Total: {len(TEST_URLS)} URLs  |  ✅ {counts['SAFE']}  ⚠️  {counts['WARNING']}  🚨 {counts['MALICIOUS']}  🛑 {counts['BLOCKED']}")

  #  VERDICT      CONF    CNN   BERT  G  RULE                              URL
---------------------------------------------------------------------------------------------------------------------------------------
  1  ✅ SAFE      100.0%    -      -    L1-whitelist                      https://www.google.com
  2  ✅ SAFE      100.0%    -      -    L1-whitelist                      https://hnb.lk/personal/internet-banking
  3  ✅ SAFE      100.0%    -      -    L1-whitelist                      https://www.dialog.lk/en/mobile-broadband
  4  🚨 MALICIOUS  97.0%  0.941  1.000  Rule 0 (hyphen-stuffed brand imp  http://paypal-secure-login.xyz/verify
  5  ⚠️  WARNING    60.0%  0.835  0.997  Rule 4 (both agree: phishing) ->  https://nawferadnan15.wixsite.com/ad-portfolio
  6  ✅ SAFE      100.0%    -      -    L1-whitelist                      https://learn.zoom.us/j/97129300588?pwd=7EertfewlCrSZaNlavG5L73Lt
  7  🚨 MALICIOUS  99.4%  0.680  0.994  Rule 4 (both agree: phishing)     http://192.168.

In [12]:
# ── Cell 9: Single URL — change this and press Shift+Enter ────────

check_url(
    
    'https://lihi1.me/eRImR'


    )


  🚨 MALICIOUS  (99.4% confidence)
  URL        : https://lihi1.me/eRImR
  Layer      : L2-ML
  Rule       : Rule 4 (both agree: phishing)
  Reason     : This link shows multiple signs of a phishing or malware attack
  CNN        : safe=0.384  malicious=0.616  (real TFLite)
  URLBert    : p_phishing=0.9938
  Features   : is_https=1.0
  Time       : 212.3 ms

